In [39]:
!pip install -q gradio pandas transformers torch accelerate bitsandbytes python-dotenv

zsh:1: command not found: pip


In [ ]:
import os
import json
import random
from datetime import datetime

import torch
import pandas as pd
import numpy as np
import gradio as gr
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from huggingface_hub import login
from dotenv import load_dotenv

load_dotenv()

# Constants
BANKS = ["Access", "GTBank", "Opay", "First", "Zenith", "Fidelity"]
STATES = ["Lagos", "Abuja", "Kano", "Rivers", "Oyo", "Kaduna", "Delta", "Enugu"]
FIRST_NAMES = ["Ade", "Chidi", "Musa", "Olu", "Ngozi", "Femi", "Zainab", "Emeka"]
LAST_NAMES = ["Okafor", "Balogun", "Abubakar", "Okonkwo", "Adebayo", "Musa"]
BUSINESSES = ["Restaurant", "Fashion", "Tech", "Agriculture", "Retail"]

In [41]:
class NigerianGenerator:
    def phone(self): return random.choice(["070","080","090"]) + ''.join([str(random.randint(0,9)) for _ in range(8)])
    def name(self): return f"{random.choice(FIRST_NAMES)} {random.choice(LAST_NAMES)}"
    def customer(self): return {"name": self.name(), "phone": self.phone(), "state": random.choice(STATES), "bank": random.choice(BANKS)}
    def business(self): return {"name": f"{random.choice(['Prime','Royal'])} {random.choice(BUSINESSES)}", "type": random.choice(BUSINESSES), "phone": self.phone(), "state": random.choice(STATES)}

gen = NigerianGenerator()

In [46]:
# Cell 4: Multiple Models with token from .env
import os
from dotenv import load_dotenv
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

load_dotenv()
HF_TOKEN = os.getenv("HUGGINGFACE_TOKEN")

# Available models
models = {
    "Phi-4-mini (Fast)": "microsoft/Phi-4-mini-instruct",
    "Llama-3.2-3B (Balanced)": "meta-llama/Llama-3.2-3B-Instruct",
    "Qwen-Coder (Code)": "Qwen/Qwen2.5-Coder-7B-Instruct",
    "Phi-3.5 (Legacy)": "microsoft/Phi-3.5-mini-instruct"
}

# Select model (change this line to use different model)
SELECTED_MODEL = models["Phi-4-mini (Fast)"]  # Default

try:
    tokenizer = AutoTokenizer.from_pretrained(SELECTED_MODEL, token=HF_TOKEN)
    model = AutoModelForCausalLM.from_pretrained(
        SELECTED_MODEL,
        quantization_config=BitsAndBytesConfig(load_in_4bit=True),
        device_map="auto",
        token=HF_TOKEN
    )
    print(f"✅ Loaded: {SELECTED_MODEL}")
except Exception as e:
    print(f"⚠️ Failed to load {SELECTED_MODEL}: {e}")
    model = None

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/15.5M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/249 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/587 [00:00<?, ?B/s]

⚠️ Failed to load microsoft/Phi-4-mini-instruct: No package metadata was found for bitsandbytes


In [47]:
def generate(prompt):
    if not model: return gen.customer()
    
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    
    outputs = model.generate(**inputs, max_new_tokens=150)
    return tokenizer.decode(outputs[0])

# Test different prompts per model
def generate_diverse(dtype):
    prompts = {
        "Customers": [
            "Generate a Lagos businessman",
            "Generate a young Nigerian professional",
            "Generate a Nigerian market trader"
        ],
        "Businesses": [
            "Generate a Nigerian tech startup",
            "Generate a Lagos restaurant",
            "Generate a Nigerian fashion brand"
        ]
    }
    return generate(random.choice(prompts[dtype]))

In [48]:
def create_dataset(dtype, n, use_llm):
    data = [gen.customer() if dtype=="Customers" else gen.business() for _ in range(n)]
    
    if use_llm and model:
        for i in range(0, n, 2):  # Every other record
            data[i] = generate_diverse(dtype)
    
    df = pd.DataFrame(data)
    df.insert(0, "id", [f"NG-{i+1:04d}" for i in range(n)])
    return df

In [49]:
with gr.Blocks(title="Nigerian Data Generator") as demo:
    gr.Markdown("# 🇳🇬 Nigerian Data Generator")
    
    with gr.Row():
        with gr.Column():
            dtype = gr.Dropdown(["Customers", "Businesses"], label="Type")
            n = gr.Slider(1, 50, 10, label="Records")
            llm = gr.Checkbox(label="Use LLM", value=True)
            btn = gr.Button("Generate", variant="primary")
        
        out = gr.Dataframe()
    
    btn.click(lambda d,n,l: create_dataset(d,int(n),l), [dtype,n,llm], out)

demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://018457bb6ab9ca15bb.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
